# 02 · Model and Training

## Colab setup

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/UNIBO/Deep Learning/DL_3C")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, str(PROJECT_DIR))
else:
    PROJECT_DIR = Path.cwd()

IMAGES_DIR = PROJECT_DIR / "images"
MODELS_DIR = PROJECT_DIR / "models"
IMAGES_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print("Working folder:", Path.cwd())
print("Files:", os.listdir())

In [ ]:
import pickle
import random
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

## Settings

In [ ]:
SEED = 42
DIM = 128
ROUNDS = 26
BATCH_SIZE = 32
EPOCHS = 30
LR = 1e-4
CLIP = 0.65

random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Load data

In [ ]:
with open("train.pkl", "rb") as f:
    train_data = pickle.load(f)

with open("val.pkl", "rb") as f:
    val_data = pickle.load(f)

print("Train:", len(train_data))
print("Val:  ", len(val_data))

## Model file

In [ ]:
%%writefile neurosat.py
import torch
import torch.nn as nn


def make_batch(problems):
    edge_lit = []
    edge_clause = []
    flip = []
    lit_graph = []
    labels = []
    lit_offset = 0
    clause_offset = 0

    for g, p in enumerate(problems):
        n = p["n_vars"]
        for c, clause in enumerate(p["clauses"]):
            for lit in clause:
                if lit > 0:
                    idx = lit - 1
                else:
                    idx = n - lit - 1
                edge_lit.append(lit_offset + idx)
                edge_clause.append(clause_offset + c)
        flip += [lit_offset + n + i for i in range(n)]
        flip += [lit_offset + i for i in range(n)]
        lit_graph += [g] * (2 * n)
        labels.append(p["label"])
        lit_offset += 2 * n
        clause_offset += len(p["clauses"])

    return {
        "edge_lit": torch.tensor(edge_lit),
        "edge_clause": torch.tensor(edge_clause),
        "flip": torch.tensor(flip),
        "lit_graph": torch.tensor(lit_graph),
        "labels": torch.tensor(labels, dtype=torch.float),
        "n_lits": lit_offset,
        "n_clauses": clause_offset,
        "n_graphs": len(problems),
    }


def to_device(batch, device):
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}


def mlp(dim, out_dim):
    return nn.Sequential(
        nn.Linear(dim, dim),
        nn.ReLU(),
        nn.Linear(dim, dim),
        nn.ReLU(),
        nn.Linear(dim, out_dim),
    )


class NeuroSAT(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.L_init = nn.Parameter(torch.randn(1, dim) / dim ** 0.5)
        self.C_init = nn.Parameter(torch.randn(1, dim) / dim ** 0.5)
        self.L_msg = mlp(dim, dim)
        self.C_msg = mlp(dim, dim)
        self.L_update = nn.LSTMCell(2 * dim, dim)
        self.C_update = nn.LSTMCell(dim, dim)
        self.L_vote = mlp(dim, 1)

    def forward(self, batch, rounds, return_literals=False):
        n_lits = batch["n_lits"]
        n_clauses = batch["n_clauses"]
        edge_lit = batch["edge_lit"]
        edge_clause = batch["edge_clause"]
        flip = batch["flip"]
        device = self.L_init.device

        L_h = self.L_init.repeat(n_lits, 1)
        L_c = torch.zeros_like(L_h)
        C_h = self.C_init.repeat(n_clauses, 1)
        C_c = torch.zeros_like(C_h)

        for _ in range(rounds):
            messages = self.L_msg(L_h)[edge_lit]
            C_in = torch.zeros(n_clauses, self.dim, device=device).index_add(0, edge_clause, messages)
            C_h, C_c = self.C_update(C_in, (C_h, C_c))

            messages = self.C_msg(C_h)[edge_clause]
            L_in = torch.zeros(n_lits, self.dim, device=device).index_add(0, edge_lit, messages)
            L_h, L_c = self.L_update(torch.cat([L_in, L_h[flip]], dim=1), (L_h, L_c))

        votes = self.L_vote(L_h).squeeze(1)
        lit_graph = batch["lit_graph"]
        totals = torch.zeros(batch["n_graphs"], device=device).index_add(0, lit_graph, votes)
        counts = torch.zeros(batch["n_graphs"], device=device).index_add(0, lit_graph, torch.ones_like(votes))
        logits = totals / counts

        if return_literals:
            return logits, L_h
        return logits

In [ ]:
from neurosat import NeuroSAT, make_batch, to_device

## Quick check

In [ ]:
model = NeuroSAT(DIM).to(device)

batch = to_device(make_batch(train_data[:4]), device)
logits = model(batch, ROUNDS)

print("Logits:", logits)
print("Labels:", batch["labels"])
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Batches

In [ ]:
def batches(data, batch_size, shuffle):
    data = data.copy()
    if shuffle:
        random.shuffle(data)
    for i in range(0, len(data), batch_size):
        yield to_device(make_batch(data[i:i + batch_size]), device)


val_batches = list(batches(val_data, BATCH_SIZE, False))

## Training setup

In [ ]:
model = NeuroSAT(DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.BCEWithLogitsLoss()

## Evaluation

In [ ]:
def evaluate(model, batch_list):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in batch_list:
            logits = model(batch, ROUNDS)
            labels = batch["labels"]
            total_loss += loss_fn(logits, labels).item() * len(labels)
            correct += ((logits > 0).float() == labels).sum().item()
            total += len(labels)

    return total_loss / total, correct / total

## Training loop

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_acc = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in batches(train_data, BATCH_SIZE, True):
        logits = model(batch, ROUNDS)
        labels = batch["labels"]
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()

        total_loss += loss.item() * len(labels)
        correct += ((logits > 0).float() == labels).sum().item()
        total += len(labels)

    train_loss = total_loss / total
    train_acc = correct / total
    val_loss, val_acc = evaluate(model, val_batches)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), MODELS_DIR / "neurosat.pt")

    print(f"Epoch {epoch:2d} | train loss {train_loss:.4f} acc {train_acc:.3f} | val loss {val_loss:.4f} acc {val_acc:.3f}")

print(f"Best validation accuracy: {best_acc:.3f}")

## Learning curves

In [ ]:
epochs = list(range(1, len(history["train_loss"]) + 1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.lineplot(x=epochs, y=history["train_loss"], label="Train", ax=axes[0])
sns.lineplot(x=epochs, y=history["val_loss"], label="Validation", ax=axes[0])
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

sns.lineplot(x=epochs, y=history["train_acc"], label="Train", ax=axes[1])
sns.lineplot(x=epochs, y=history["val_acc"], label="Validation", ax=axes[1])
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")

plt.tight_layout()
plt.savefig(IMAGES_DIR / "learning_curves.png", dpi=150)
plt.show()

## Save history

In [ ]:
with open("history.pkl", "wb") as f:
    pickle.dump(history, f)

print("Saved neurosat.pt, history.pkl and learning_curves.png")